# Development of cell match across scans

In [1]:
import pandas as pd

import datajoint as dj
import matplotlib.pyplot as plt
%matplotlib inline
#from pipeline.utils import registration
import numpy as np
from neuro_data.static_images.data_schemas import stimulus, fuse, meso, experiment
stack = dj.create_virtual_module('stack', 'pipeline_stack')
from staticnet_analyses import multi_mei, closed_loop
from staticnet_analyses.closed_loop import *
from itertools import count

import seaborn as sns

dj.config['display.limit'] = 50

Connecting eywalker@10.28.0.34:3306


/src/static-networks/staticnet_analyses/multi_mei.py:1519: UserWarning: Use of this table is deprecated. It is kept only for record keeping purpose
  warnings.warn('Use of this table is deprecated. It is kept only for record keeping purpose')


In [2]:
ProximityCellMatch.populate(display_progress=True)

0it [00:00, ?it/s]


In [10]:
meso.StackCoordinates & 'animal_id = 20892'

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,pipe_version,field,channel,segmentation_method,stack_session session index for the mouse,stack_idx id of the stack,volume_id id of this volume,stack_channel,scan_channel,registration_method method used for registration
20892,3,11,1,1,1,6,3,20,1,1,1,5
20892,3,11,1,2,1,6,3,20,1,1,1,5
20892,3,11,1,3,1,6,3,20,1,1,1,5
20892,3,11,1,4,1,6,3,20,1,1,1,5
20892,3,11,1,5,1,6,3,20,1,1,1,5
20892,3,11,1,6,1,6,3,20,1,1,1,5
20892,3,11,1,7,1,6,3,20,1,1,1,5
20892,3,11,1,8,1,6,3,20,1,1,1,5
20892,3,11,1,9,1,6,3,20,1,1,1,5
20892,3,11,1,10,1,6,3,20,1,1,1,5


In [5]:
ProximityCellMatch() & (StaticMultiDataset.Member & 'group_id = 21')

animal_id id number,src_session session index for the mouse,src_scan_idx number of TIFF stack file,pipe_version,src_field,channel,segmentation_method,stack_session session index for the mouse,stack_idx id of the stack,volume_id id of this volume,stack_channel,scan_channel,registration_method method used for registration,session session index for the mouse,scan_idx number of TIFF stack file


In [3]:
BestProximityCellMatch.populate(display_progress=True)

100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


In [22]:
(ClosedLoopScan & 'loop_group > 0') - BestProximityCellMatch()

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,loop_group used to group related scans,day day number in the closed loop starting with 1,mei_source whether it was used to generate MEI,stim_type name of the stim type
20457,5,9,1,1,1,imagenet
20505,10,14,2,1,1,imagenet


In [6]:
schema = dj.schema('edgar_backup')

In [9]:
@schema
class BPCMBackup(dj.Manual):
    definition = """
    animal_id            : int                          # id number
    src_session          : smallint                     # session index for the mouse
    src_scan_idx         : smallint                     # number of TIFF stack file
    pipe_version         : smallint                     # 
    segmentation_method  : tinyint                      # 
    src_unit_id          : int                          # unique per scan & segmentation method
    session              : smallint                     # session index for the mouse
    scan_idx             : smallint                     # number of TIFF stack file
    ---
    unit_id              : int                          # unique per scan & segmentation method
    match_freq           : int                          # how many times it was matched
    total_stacks         : int                          # number of stacks used to match
    mean_distance        : float                        # average match distance (in um)
    """

In [15]:
BestProximityCellMatch * BPCMBackup.proj(old_unit_id='unit_id')

animal_id id number,src_session session index for the mouse,src_scan_idx number of TIFF stack file,pipe_version,segmentation_method,src_unit_id unique per scan & segmentation method,session session index for the mouse,scan_idx number of TIFF stack file,unit_id unique per scan & segmentation method,match_freq how many times it was matched,total_stacks number of stacks used to match,mean_distance average match distance (in um),old_unit_id unique per scan & segmentation method
